*Module 5 of 9*

> **¿Prefieres español?** Abre [`05_de_pixeles_a_parcelas.ipynb`](../es/05_de_pixeles_a_parcelas.ipynb) — es el mismo módulo, en español.


# 🧩 Module 5 — From pixels to parcels: segmentation

🧭 **Objectives** — understand why we classify **parcels** instead of
individual pixels, get the intuition behind **k-means** and the **Shepherd**
segmentation algorithm, run it on your tile with `shepherd-wasm`, and see the
field carved into homogeneous objects.

📚 **Why not pixels?** A single field is hundreds of 30 m pixels. Classifying
each pixel alone gives "salt-and-pepper" noise — stray misclassified dots
inside an obviously uniform field. **Object-based** classification first
groups neighboring, spectrally-similar pixels into **segments** (parcels),
then classifies each *parcel* as a whole. Cleaner maps, and it matches how
agriculture actually works: decisions happen per field, not per pixel.

📚 **Shepherd segmentation** (Shepherd et al., 2019) does it in three steps:
1. **k-means** groups pixels into a handful of spectral "families".
2. **Clumping**: connected pixels of the same family become one segment.
3. **Elimination**: segments smaller than a threshold are merged into their
   most similar neighbor, so no sliver is left behind.

We use **`shepherd-wasm`**, a pure NumPy/SciPy port that runs in the browser
(the desktop pipeline uses the numba-accelerated `pyshepseg` — same
algorithm, different engine; you will see this in Module 9).

![segmentation](../../anim/en/05_segmentation.svg)


## The k-means intuition (tiny demo)

Before segmenting the real tile, feel what k-means does: it sorts points into
`k` groups by similarity. Here we sort a handful of fake pixels (each with an
NDVI and a water index) into 3 spectral families. No geography yet — just
"which pixels resemble which".


In [ ]:
import numpy as np
from sklearn.cluster import KMeans

# 9 fake pixels: [NDVI, water index]. Three natural groups.
pixels = np.array([[0.8, 0.1], [0.82, 0.12], [0.79, 0.09],   # dense crop
                   [0.2, 0.1], [0.18, 0.08], [0.22, 0.11],   # bare soil
                   [-0.3, 0.6], [-0.28, 0.62], [-0.31, 0.58]])# water
labels = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(pixels)
print("Pixel -> family:", labels)
print("k-means found the 3 groups without being told what they are.")

In [ ]:
# Get the workshop tile (a few MB; cached after the first download)
import os, sys

async def get_file(name):
    for cand in (f"files/{name}", name, f"../files/{name}", f"../../files/{name}"):
        if os.path.exists(cand):
            return cand
    dest = f"/tmp/{name}"
    if not os.path.exists(dest):
        url = f"https://raw.githubusercontent.com/abxda/portable-geocrop/main/files/{name}"
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            resp = await pyfetch(url)
            open(dest, "wb").write(await resp.bytes())
        else:
            import urllib.request
            urllib.request.urlretrieve(url, dest)
    return dest

TILE = await get_file("crop_tile_384.tif")
print("Tile ready:", TILE)

## Segment the real tile

`shepherd-wasm` needs `scipy.ndimage` and `sklearn.cluster` imported first
(a Pyodide quirk — its auto-loader cannot see the internal imports).
`numClusters` sets how many spectral families k-means seeds; `minSegmentSize`
is the smallest parcel allowed (smaller ones get merged away). This takes
roughly 30–60 seconds in the browser — watch for the `[*]`.


In [ ]:
# Pyodide: import these BEFORE shepherd_wasm so its internals resolve
import scipy.ndimage, sklearn.cluster
import shepherd_wasm, time, rasterio

with rasterio.open(TILE) as src:
    img = src.read()               # (13, 384, 384)

t0 = time.time()
result = shepherd_wasm.doShepherdSegmentation(
    img, numClusters=30, minSegmentSize=50, imgNullVal=0, fixedKMeansInit=True)
seg = result.segimg.astype(np.int32)
n_seg = int(seg.max())
print(f"{n_seg} parcels found in {time.time()-t0:.1f} s")

In [ ]:
import matplotlib.pyplot as plt
from scipy import ndimage

rgb = np.clip(np.dstack([img[2], img[1], img[0]]) / 3000.0, 0, 1)
# Draw parcel boundaries in yellow over the true-color image
edges = (ndimage.maximum_filter(seg, size=2) != ndimage.minimum_filter(seg, size=2))
vis = rgb.copy(); vis[edges] = [1, 1, 0]

plt.figure(figsize=(8, 8)); plt.imshow(vis)
plt.title(f"{n_seg} parcels (yellow = boundaries)"); plt.axis("off"); plt.show()
print("Each yellow-bordered patch is one object we will classify.")

## Experiment

Change `numClusters` (try 15 or 50) and `minSegmentSize` (try 20 or 120) and
re-run the two cells above. Fewer clusters / bigger minimum = larger, coarser
parcels; more clusters / smaller minimum = finer detail but more fragments.
There is no single "right" answer — it depends on the size of the fields you
want to capture.


## 🧪 Check yourself

**What is "salt-and-pepper" noise, and how does segmentation cure it?**

<details><summary>Show answer</summary>

It is scattered, individually misclassified pixels inside a field that is
really uniform. Segmentation groups the field's pixels into one object and
classifies the object as a whole, so a few odd pixels can't speckle the map.

</details>

**In Shepherd, what does `minSegmentSize` control, and what happens to
parcels below it?**

<details><summary>Show answer</summary>

It is the smallest allowed segment size. Segments smaller than it are merged
into their spectrally most similar neighbor during the elimination step, so
no tiny slivers survive.

</details>


## 🔭 Go deeper

Optional: these bilingual concept cards expand what you just learned
(prerequisite chains, lineage to fundamentals, curated references):

- [Image segmentation](https://abxda.github.io/rs-learning-audio/?id=image-segmentation)
- [Segmentation (concept)](https://abxda.github.io/rs-learning-audio/?id=segmentation)
- [Object-based classification](https://abxda.github.io/rs-learning-audio/?id=object-based-classification)
- [Clustering / k-means](https://abxda.github.io/rs-learning-audio/?id=clustering)
- [Pixel vs object classification](https://abxda.github.io/rs-learning-audio/?id=pixel-classification)



---

[← Previous · Module 4 — Vegetation indices](04_vegetation_indices.ipynb) · [Next → · Module 6 — Parcels become a table + the ground truth](06_features_and_labels.ipynb)
